In [1]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from skimage.filters import threshold_otsu, median
from skimage.morphology import (
    binary_closing,
    binary_opening,
    disk,
    ball,
    remove_small_objects,
    binary_dilation,
    binary_erosion
)
from skimage.measure import label, regionprops, find_contours
from scipy import ndimage
from sklearn.metrics import jaccard_score
from skimage.measure import label as sk_label
import os

FILES_DIR = 'files'
GENERATED_DIR = 'generated_files'
os.makedirs(GENERATED_DIR, exist_ok=True)

In [2]:
# MODIFIED PARAMETERS 
FRUIT_THRESH_ADJUST = 0.45   # More stringent threshold for the fruit
SEED_PERCENTILE = 12         # Lower percentile for seeds
MIN_SEED_SIZE_3D = 5         # Smaller minimum seed size
CORE_SIZE_THRESH = 40       # Minimum core size
MORPH_DISK_SIZE = 2          # Smaller structural element
VOXEL_SIZE = (0.2344, 0.2344, 0.5)
SLICE_STEP = 10
SEED_SEARCH_MARGIN = 9      # Seed search margin [mm]
FRUIT_CLOSING_SIZE = 1
start_slice = 20
end_slice = 110
CORE_CLOSING_SIZE = 1        # Sphere size for 3D closing of core (from previous code)

In [3]:
def load_nifti(path):
    img = nib.load(path)
    return img.get_fdata(), img

def save_nifti(data, path, reference_img):
    new_img = nib.Nifti1Image(data, reference_img.affine, reference_img.header)
    nib.save(new_img, path)

In [4]:
def display_slices(volume, title='', cmap='gray', slice_step=SLICE_STEP):
    num_slices = volume.shape[2]
    slices = list(range(0, num_slices, slice_step))
    if num_slices - 1 not in slices:
        slices.append(num_slices - 1)
    
    n_cols = 4
    n_rows = (len(slices) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3))
    fig.suptitle(title, fontsize=16)
    
    for i, z in enumerate(slices):
        row = i // n_cols
        col = i % n_cols
        ax = axes[row, col] if n_rows > 1 else axes[col]
        ax.imshow(volume[:, :, z], cmap=cmap)
        ax.set_title(f'Slice {z}/{num_slices}')
        ax.axis('off')
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    outpath = os.path.join(GENERATED_DIR, f'{title.replace(" ", "_")}.png')
    plt.savefig(outpath, dpi=150, bbox_inches='tight')
    plt.show()

In [5]:
def preprocess_volume(volume):
    volume = (volume - volume.min()) / (volume.max() - volume.min())
    filtered_vol = np.zeros_like(volume)
    for z in range(volume.shape[2]):
        filtered_vol[:, :, z] = median(volume[:, :, z], disk(1))
    return filtered_vol

In [6]:
def segment_fruit_3d(volume):
    mask = np.zeros_like(volume, dtype=bool)
    
    for z in range(volume.shape[2]):
        slice = volume[:, :, z]
        thresh = threshold_otsu(slice) * FRUIT_THRESH_ADJUST
        slice_mask = slice > thresh
        
        # Less aggressive morphological operations
        slice_mask = binary_closing(slice_mask, disk(MORPH_DISK_SIZE))
        slice_mask = ndimage.binary_fill_holes(slice_mask)
        
        mask[:, :, z] = slice_mask
    
    # Fill holes in 3D space
    mask = ndimage.binary_fill_holes(mask)
    mask = remove_small_objects(mask, min_size=100)
    return mask

In [7]:
def segment_core_3d(volume, fruit_mask):

    #Segment the fruit core as a single large dark region (from code 2)
    core_candidates = np.zeros_like(volume, dtype=bool)
    
    for z in range(volume.shape[2]):
        if not np.any(fruit_mask[:, :, z]):
            continue
            
        slice = volume[:, :, z]
        fruit_slice = fruit_mask[:, :, z]
        
        # Threshold for dark regions (core and seeds)
        dark_thresh = np.percentile(slice[fruit_slice], SEED_PERCENTILE)
        dark_mask = (slice < dark_thresh) & fruit_slice
        
        core_candidates[:, :, z] = dark_mask
    
    # Label in 3D and find the largest object
    labeled = label(core_candidates)
    regions = regionprops(labeled)
    
    if not regions:
        return np.zeros_like(volume, dtype=bool)
    
    # Find the largest region
    largest_region = max(regions, key=lambda r: r.area)
    
    # Create mask only for the core
    core_mask = (labeled == largest_region.label)
    
    # Apply 3D closing operation using spherical structuring element
    core_mask = binary_closing(core_mask, ball(CORE_CLOSING_SIZE))
    
    # Remove small objects (keep only core)
    core_mask = remove_small_objects(core_mask, min_size=CORE_SIZE_THRESH)
    
    return core_mask

In [8]:
def segment_seeds_3d(volume, fruit_mask, core_mask):
    
#Segment seeds excluding the core and restricted to the interior of the fruit (from code 1)
    seeds_mask = np.zeros_like(volume, dtype=bool)
    # Seed search area (fruit eroded by margin and excluding core)
    seed_search_area = np.zeros_like(volume, dtype=bool)
    
    # Convert margin from mm to pixels (average)
    avg_pixel_size = np.mean(VOXEL_SIZE[:2])
    margin_pixels = int(SEED_SEARCH_MARGIN / avg_pixel_size)
    
    # For each slice:
    for z in range(volume.shape[2]):
        if not np.any(fruit_mask[:, :, z]):
            continue

        # Fruit mask in the current slice
        fruit_slice = fruit_mask[:, :, z]
        # Erode fruit mask to create a smaller region (margin from boundary)
        eroded_fruit = binary_erosion(fruit_slice, disk(margin_pixels))
        # Search area: inside eroded fruit and outside the core
        search_area = eroded_fruit & ~core_mask[:, :, z]
        seed_search_area[:, :, z] = search_area

        # Threshold only within the search area
        slice = volume[:, :, z]
        fruit_pixels = slice[search_area]
        if len(fruit_pixels) == 0:
            continue
            
        seed_thresh = np.percentile(fruit_pixels, SEED_PERCENTILE)
        seed_mask = (slice < seed_thresh) & search_area

        # Gentle cleanup
        seed_mask = binary_opening(seed_mask, disk(1))
        seed_mask = remove_small_objects(seed_mask, min_size=3)

        seeds_mask[:, :, z] = seed_mask

    # 3D cleanup
    seeds_mask = seeds_mask.astype(bool)
    seeds_mask = remove_small_objects(seeds_mask, min_size=MIN_SEED_SIZE_3D)
    
    # Remove any remaining core artifacts
    seeds_mask = seeds_mask & ~core_mask
    
    return seeds_mask, seed_search_area

In [9]:
# NEW: Colors for segmentation
SEED_COLOR = [1, 0, 0, 0.7]  # Red with transparency
CORE_COLOR = [1, 0.5, 0, 0.7]  # Orange with transparency

In [10]:
def visualize_seed_slices(original, seeds_mask, search_area, layers, title='Seed analysis'):
    
    #Show selected slices with the seed search area outlined in red.
     
    n_rows = len(layers)
    fig, axes = plt.subplots(n_rows, 3, figsize=(15, 5 * n_rows))
    fig.suptitle(title, fontsize=16)
    
    if n_rows == 1:
        axes = np.array([axes])
    
    for i, z in enumerate(layers):
        # Original slice
        axes[i, 0].imshow(original[:, :, z], cmap='gray')
        axes[i, 0].set_title(f'Original - Slice {z + start_slice}')
        axes[i, 0].axis('off')
        
        # Seed mask with search boundary
        axes[i, 1].imshow(seeds_mask[:, :, z], cmap='gray')
        
        # Draw red contour for search area
        search_contour = find_contours(search_area[:, :, z], 0.5)
        for contour in search_contour:
            axes[i, 1].plot(contour[:, 1], contour[:, 0], linewidth=1.5, color='red')
        
        axes[i, 1].set_title('Seeds with search area')
        axes[i, 1].axis('off')
        
        # Overlay seeds on original
        axes[i, 2].imshow(original[:, :, z], cmap='gray')
        axes[i, 2].imshow(
            np.ma.masked_where(~seeds_mask[:, :, z], seeds_mask[:, :, z]),
            cmap='autumn', alpha=0.7
        )
        for contour in search_contour:
            axes[i, 2].plot(contour[:, 1], contour[:, 0], linewidth=1.5, color='red')
        axes[i, 2].set_title('Overlay with search contour')
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    outpath = os.path.join(GENERATED_DIR, f'{title.replace(" ", "_")}.png')
    plt.savefig(outpath, dpi=300, bbox_inches='tight')
    plt.show()

In [11]:
def visualize_overlay_slices(original, seeds_mask, search_area, layers, title='Segmentation overlay on original image'):

    #Show selected slices with segmentation overlay and search area contours.
    
    n_cols = len(layers)
    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5))
    fig.suptitle(title, fontsize=16)
    
    if n_cols == 1:
        axes = np.array([axes])
    
    for i, z in enumerate(layers):
        axes[i].imshow(original[:, :, z], cmap='gray')
        axes[i].imshow(
            np.ma.masked_where(~seeds_mask[:, :, z], seeds_mask[:, :, z]),
            cmap='autumn', alpha=0.7
        )
        search_contour = find_contours(search_area[:, :, z], 0.5)
        for contour in search_contour:
            axes[i].plot(contour[:, 1], contour[:, 0], linewidth=1.5, color='red')
        axes[i].set_title(f'Slice {z + start_slice}')
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.85)
    outpath = os.path.join(GENERATED_DIR, f'{title.replace(" ", "_")}.png')
    plt.savefig(outpath, dpi=300, bbox_inches='tight')
    plt.show()

In [12]:
def visualize_slice_overview(volume, layers, title='Slice overview', cmap='gray'):

    #Display selected slices in a grid layout
    
    n_cols = 4
    n_rows = (len(layers) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3))
    fig.suptitle(title, fontsize=16)
    
    for i, z in enumerate(layers):
        row = i // n_cols
        col = i % n_cols
        ax = axes[row, col] if n_rows > 1 else axes[col]
        ax.imshow(volume[:, :, z], cmap=cmap)
        ax.set_title(f'Slice {z + start_slice}')
        ax.axis('off')
    
    # Hide empty subplots
    total_plots = n_rows * n_cols
    for j in range(len(layers), total_plots):
        row = j // n_cols
        col = j % n_cols
        if n_rows > 1:
            axes[row, col].axis('off')
        else:
            axes[col].axis('off')
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    outpath = os.path.join(GENERATED_DIR, f'{title.replace(" ", "_")}.png')
    plt.savefig(outpath, dpi=150, bbox_inches='tight')
    plt.show()

In [13]:
def calculate_metrics(fruit_mask, seeds_mask, core_mask):
    voxel_vol = VOXEL_SIZE[0] * VOXEL_SIZE[1] * VOXEL_SIZE[2]
    
    fruit_vol = np.sum(fruit_mask) * voxel_vol
    seeds_vol = np.sum(seeds_mask) * voxel_vol
    core_vol = np.sum(core_mask) * voxel_vol

    # Count seeds excluding the core - UPDATED METHOD
    labeled_seeds = sk_label(seeds_mask & ~core_mask, connectivity=3)
    num_seeds = len(np.unique(labeled_seeds)) - 1  # subtract background (label 0)
    
    ratio = seeds_vol / fruit_vol
    
    return {
        "Number of seeds": num_seeds,
        "Fruit volume [mm3]": round(fruit_vol, 2),
        "Seeds volume [mm3]": round(seeds_vol, 2),
        "Core volume [mm3]": round(core_vol, 2),
        "Seed/fruit ratio": round(ratio, 4),
        "Seed search margin [mm]": SEED_SEARCH_MARGIN,
        "Core closing size": CORE_CLOSING_SIZE
    }

In [14]:
def main(input_filename):
    
    #Process a file by name located in the FILES_DIR folder.

    # Build full path (search under files directory)
    if os.path.isabs(input_filename):
        input_path = input_filename
    else:
        input_path = os.path.join(FILES_DIR, input_filename)
    
    # 1. Load data without cropping
    data, img_obj = load_nifti(input_path)
    original_shape = data.shape
    print(f"Original size: {original_shape}")
    
    # 2. Create full-size masks
    fruit_mask_full = np.zeros(original_shape, dtype=bool)
    core_mask_full = np.zeros(original_shape, dtype=bool)
    seeds_mask_full = np.zeros(original_shape, dtype=bool)
    seed_search_area_full = np.zeros(original_shape, dtype=bool)
    
    # 3. Select processing range (slices 20-110)
    processing_slices = slice(start_slice, end_slice + 1)
    processing_data = data[:, :, processing_slices]
    print(f"Processing range size: {processing_data.shape}")
    
    # 4. Preprocessing and segmentation only on selected slices
    processed_vol = preprocess_volume(processing_data)
    fruit_mask = segment_fruit_3d(processed_vol)
    core_mask = segment_core_3d(processed_vol, fruit_mask)
    seeds_mask, seed_search_area = segment_seeds_3d(processed_vol, fruit_mask, core_mask)
    
    # 5. Write results into full-size masks
    fruit_mask_full[:, :, processing_slices] = fruit_mask
    core_mask_full[:, :, processing_slices] = core_mask
    seeds_mask_full[:, :, processing_slices] = seeds_mask
    seed_search_area_full[:, :, processing_slices] = seed_search_area
    
    # 6. Select layers for visualization (in original numbering)
    num_slices = processing_data.shape[2]
    key_layers = [0, num_slices//4, num_slices//2, 3*num_slices//4, num_slices-1]
    key_layers = sorted(set(min(z, num_slices-1) for z in key_layers))
    key_layers_original = [z + start_slice for z in key_layers]
    detailed_layers_original = [key_layers_original[1], key_layers_original[2], key_layers_original[3]]
    
    # 7. Visualizations on full data
    visualize_slice_overview(data, key_layers_original, title='Kiwi MRI overview')
    visualize_slice_overview(seeds_mask_full, key_layers_original, title='Seed segmentation', cmap='gray')
    
    # 8. Detailed visualizations
    visualize_seed_slices(
        data,
        seeds_mask_full,
        seed_search_area_full,
        detailed_layers_original,
        title='Kiwi seed analysis'
    )
    
    visualize_overlay_slices(
        data,
        seeds_mask_full,
        seed_search_area_full,
        detailed_layers_original,
        title='Segmentation overlay on original image'
    )
    
    # 9. Compute segmentation metrics
    metrics = calculate_metrics(fruit_mask_full, seeds_mask_full, core_mask_full)
    
    # 10. Baseline evaluation - added
    # Load ground truth mask from files folder
    gt_path = os.path.join(FILES_DIR, 'kiwi_bergen_wzorzec.nii')
    gt_mask, _ = load_nifti(gt_path)
    
    # Prepare masks for evaluation (only processed slices)
    gt_mask_processed = gt_mask[:, :, processing_slices].astype(bool)
    our_seeds_processed = seeds_mask.copy()
    
    # Calculate Jaccard index
    jaccard = jaccard_score(
        gt_mask_processed.ravel(),
        our_seeds_processed.ravel(),
        average='binary'  # for binary classification
    )
    
    # Count seeds in ground truth
    gt_labeled = sk_label(gt_mask_processed)
    gt_seed_count = len(np.unique(gt_labeled)) - 1  # subtract background
    
    # Add evaluation metrics to results
    metrics["Jaccard index (seeds)"] = round(jaccard, 4)
    metrics["Seed count (ground truth)"] = gt_seed_count
    metrics["Seed count difference"] = abs(metrics["Number of seeds"] - gt_seed_count)
    
    # 11. Save results to generated_files folder
    save_nifti(fruit_mask_full.astype(np.uint8), os.path.join(GENERATED_DIR, 'mask_fruit.nii.gz'), img_obj)
    save_nifti(core_mask_full.astype(np.uint8), os.path.join(GENERATED_DIR, 'mask_core.nii.gz'), img_obj)
    save_nifti(seeds_mask_full.astype(np.uint8), os.path.join(GENERATED_DIR, 'mask_seeds.nii.gz'), img_obj)
    save_nifti(seed_search_area_full.astype(np.uint8), os.path.join(GENERATED_DIR, 'seed_search_area.nii.gz'), img_obj)
    
    # 12. Report - updated
    with open(os.path.join(GENERATED_DIR, 'report.txt'), 'w') as f:
        f.write("Kiwi MRI analysis results:\n")
        f.write("========================\n\n")
        f.write("SEGMENTATION METRICS:\n")
        for k, v in metrics.items():
            if "Jaccard" in k or "difference" in k:
                continue
            f.write(f"{k}: {v}\n")
        
        f.write("\nBASELINE EVALUATION:\n")
        f.write(f"Jaccard index (seeds): {metrics['Jaccard index (seeds)']}\n")
        f.write(f"Seed count (ground truth): {metrics['Seed count (ground truth)']}\n")
        f.write(f"Seed count (algorithm): {metrics['Number of seeds']}\n")
        f.write(f"Seed count difference: {metrics['Seed count difference']}\n")
        f.write("\nSegmentation parameters:\n")
    
    print("\n--- ANALYSIS RESULTS ---")
    print("SEGMENTATION METRICS:")
    for k, v in metrics.items():
        if "Jaccard" in k or "difference" in k:
            continue
        print(f"{k}: {v}")
    
    print("\nBASELINE EVALUATION:")
    print(f"Jaccard index (seeds): {metrics['Jaccard index (seeds)']}")
    print(f"Seed count (ground truth): {metrics['Seed count (ground truth)']}")
    print(f"Seed count difference: {metrics['Seed count difference']}")

In [15]:
if __name__ == "__main__":
    results = main('kiwi_bergen.nii.gz')

Original size: (256, 256, 128)
Processing range size: (256, 256, 91)


C:\Users\adamk\AppData\Local\Temp\ipykernel_6008\1347740437.py:10: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  slice_mask = binary_closing(slice_mask, disk(MORPH_DISK_SIZE))
C:\Users\adamk\AppData\Local\Temp\ipykernel_6008\1347740437.py:10: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  slice_mask = binary_closing(slice_mask, disk(MORPH_DISK_SIZE))
C:\Users\adamk\AppData\Local\Temp\ipykernel_6008\1347740437.py:10: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.closing` instead.
  slice_mask = binary_closing(slice_mask, disk(MORPH_DISK_SIZE))
C:\Users\adamk\AppData\Local\Temp\ipykernel_6008\1347740437.py:10: FutureWarning: `binary_closing` is deprecated since version 0.26 and will be removed in version 0.28. Use

KeyboardInterrupt: 